# US Crime Prediction - Communities and Crime Unnormalized Dataset
---
- **Objetivo:** El presente notebook busca realizar un analisis exploratorio del dataset communities and crime unnormalized en donde se observaran las correlaciones y la estructura de los datos, ademas, se experimentara con combinaciones de los atributos y se generaran graficos estadisticos (histogramas) con el fin de identificar que features son informativos, que problemas estructurales tiene el dataset y que decisiones de preprocesamiento seran necesarias antes de entrenar un modelo.

- **Autor:** Alejandro Santos
- **Fecha:** 26/05/2026
- **Dataset:** UC Irvine Communities and Crime Unnormalized Dataset

## 1. Carga de datos

Se usa cargar_data_uci() para descargar y almacenar el dataset localmente. La funcion recoge la informacion desde la plataforma de UC Irvine.

- **Lo que se quiere verificar:** Que el archivo se descargo correctamente y que el shape (tama;o) es el esperado (2215 registros y 147 columnas)

In [19]:
from src.data_fnc.load_data import cargar_data_uci
import pandas as pd

In [20]:
df = cargar_data_uci(211)

Dataset cargado y guardado en: data\raw\dataset_211.csv


In [21]:
df.head()

,communityname,countyCode,communityCode,fold,State,pop,perHoush,pctBlack,pctWhite,pctAsian,...,burglaries,burglPerPop,larcenies,larcPerPop,autoTheft,autoTheftPerPop,arsons,arsonsPerPop,violentPerPop,nonViolPerPop
0,BerkeleyHeightstownship,39.0,5320.0,1,NJ,11980,3.10,1.37,91.78,6.50,...,14.0,114.85,138.0,1132.08,16.0,131.26,2.0,16.41,41.02,1394.59
1,Marpletownship,45.0,47616.0,1,PA,23123,2.82,0.80,95.57,3.44,...,57.0,242.37,376.0,1598.78,26.0,110.55,1.0,4.25,127.56,1955.95
2,Tigardcity,NaN,NaN,1,OR,29344,2.43,0.74,94.33,3.43,...,274.0,758.14,1797.0,4972.19,136.0,376.30,22.0,60.87,218.59,6167.51
3,Gloversvillecity,35.0,29443.0,1,NY,16656,2.40,1.70,97.35,0.50,...,225.0,1301.78,716.0,4142.56,47.0,271.93,NaN,NaN,306.64,NaN
4,Bemidjicity,7.0,5068.0,1,MN,11245,2.76,0.53,89.16,1.17,...,91.0,728.93,1060.0,8490.87,91.0,728.93,5.0,40.05,NaN,9988.79


In [22]:
print(f"Shape: {df.shape}")

Shape: (2215, 147)


In [27]:
# Eliminacion de otras columnas que eran posibles targets pero que en este caso no se utilizaran

df = df.drop(columns=['murders',
                      'murdPerPop',
                      'rapes',
                      'rapesPerPop',
                      'robbbPerPop',
                      'assaults',
                      'assaultPerPop',
                      'burglaries',
                      'burglPerPop',
                      'larcenies',
                      'larcPerPop',
                      'autoTheft',
                      'autoTheftPerPop',
                      'arsons',
                      'arsonsPerPop',
                      'violentPerPop',
                      'nonViolPerPop'])

In [30]:
print(f"Shape: {df.shape}")

Shape: (2215, 130)


## 2. Exploracion inicial del dataset

Antes de cualquier transformacion necesitamos responder las siguientes preguntas que condicionaran todo el preprocesamiento:

- Que tan complejo esta? -- Nulos criticos pueden descartar features enteras
- Existen variables categoricas?
- Las escalas son comparables? -- Determina si necesitamos StandardScaler o RobustScaler
- Que tan asimetricos son los features? -- Gran asimetria positiva (Power law distribution) necesita ser transformada logaritmicamente, si no es tan exagerada puede transformarse mediante raiz cuadrada
- Hay desbalance de clases? -- Determina si estratificamos el split o aplicamos SMOTE
- Que features estan mas correlacionados con el target? -- Determina si necesitamos talvez combinar features para que ganen capacidad predictiva

### 2.1. Estructura y nulos

In [28]:
## Configuracion de pandas para que muestre todas las columnas.
pd.set_option('display.max_columns', None)

In [ ]:
df.describe()

,countyCode,communityCode,fold,pop,perHoush,pctBlack,pctWhite,pctAsian,pctHisp,pct12-21,pct12-29,pct16-24,pct65up,persUrban,pctUrban,medIncome,pctWwage,pctWfarm,pctWdiv,pctWsocsec,pctPubAsst,pctRetire,medFamIncome,perCapInc,whitePerCap,blackPerCap,NAperCap,asianPerCap,otherPerCap,hispPerCap,persPoverty,pctPoverty,pctLowEdu,pctNotHSgrad,pctCollGrad,pctUnemploy,pctEmploy,pctEmployMfg,pctEmployProfServ,pctOccupManu,pctOccupMgmt,pctMaleDivorc,pctMaleNevMar,pctFemDivorc,pctAllDivorc,persPerFam,pct2Par,pctKids2Par,pctKids-4w2Par,pct12-17w2Par,pctWorkMom-6,pctWorkMom-18,kidsBornNevrMarr,pctKidsBornNevrMarr,numForeignBorn,pctFgnImmig-3,pctFgnImmig-5,pctFgnImmig-8,pctFgnImmig-10,pctImmig-3,pctImmig-5,pctImmig-8,pctImmig-10,pctSpeakOnlyEng,pctNotSpeakEng,pctLargHousFam,pctLargHous,persPerOccupHous,persPerOwnOccup,persPerRenterOccup,pctPersOwnOccup,pctPopDenseHous,pctSmallHousUnits,medNumBedrm,houseVacant,pctHousOccup,pctHousOwnerOccup,pctVacantBoarded,pctVacant6up,medYrHousBuilt,pctHousWOphone,pctHousWOplumb,ownHousLowQ,ownHousMed,ownHousUperQ,ownHousQrange,rentLowQ,rentMed,rentUpperQ,rentQrange,medGrossRent,medRentpctHousInc,medOwnCostpct,medOwnCostPctWO,persEmergShelt,persHomeless,pctForeignBorn,pctBornStateResid,pctSameHouse-5,pctSameCounty-5,pctSameState-5,numPolice,policePerPop,policeField,policeFieldPerPop,policeCalls,policCallPerPop,policCallPerOffic,policePerPop2,racialMatch,pctPolicWhite,pctPolicBlack,pctPolicHisp,pctPolicAsian,pctPolicMinority,officDrugUnits,numDiffDrugsSeiz,policAveOT,landArea,popDensity,pctUsePubTrans,policCarsAvail,policOperBudget,pctPolicPatrol,gangUnit,pctOfficDrugUnit,policBudgetPerPop,robberies
count,994.000000,991.000000,2215.000000,2.215000e+03,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2.215000e+03,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2214.000000,2215.000000,2.215000e+03,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2.215000e+03,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,2215.000000,343.000000,343.000000,343.000000,343.000000,3.430000e+02,3.430000e+02,343.000000,343.000000,343.000000,343.000000,343.000000,343.000000,343.000000,343.000000,343.000000,343.000000,343.000000,2215.000000,2215.000000,2215.000000,343.000000,3.430000e+02,343.000000,343.000000,2215.000000,3.430000e+02,2214.000000
mean,65.587525,45209.251261,5.494357,5.311798e+04,2.707327,9.335102,83.979819,2.670203,7.950176,14.445837,27.644840,13.975142,11.836393,4.773472e+04,70.465309,33984.696163,78.312758,0.881842,43.750935,26.409418,6.801445,15.969002,39857.055079,15603.524605,16567.698420,11541.749436,12229.191422,14227.989616,9442.765131,11018.998194,7.590853e+03,11.620537,9.186646,22.305120,23.056876,6.045242,62.021612,18.228907,24.532298,13.819165,28.209201,9.127585,30.683517,12.325300,10.812515,3.129698,74.059129,71.227255,81.865422,75.521788,60.542641,68.854795,2141.418962,3.115499,6.277274e+03,13.525693,20.421287,27.544181,34.733928,1.099124,1.697463,2.307503,2.943761,87.074993,2.405792,5.386619,3.915788,2.615842,2.740483,2.367138,66.369454,4.132438,45.405341,2.640632,1748.368849,92.933973,63.368298,2.778524,34.773887,1962.6234

In [32]:
df.info(verbose=True, show_counts=True) # verbose y show_counts para ignorar el limite de columnas

<class 'pandas.DataFrame'>
RangeIndex: 2215 entries, 0 to 2214
Data columns (total 130 columns):
 #    Column               Non-Null Count  Dtype  
---   ------               --------------  -----  
 0    communityname        2215 non-null   str    
 1    countyCode           994 non-null    float64
 2    communityCode        991 non-null    float64
 3    fold                 2215 non-null   int64  
 4    State                2215 non-null   str    
 5    pop                  2215 non-null   int64  
 6    perHoush             2215 non-null   float64
 7    pctBlack             2215 non-null   float64
 8    pctWhite             2215 non-null   float64
 9    pctAsian             2215 non-null   float64
 10   pctHisp              2215 non-null   float64
 11   pct12-21             2215 non-null   float64
 12   pct12-29             2215 non-null   float64
 13   pct16-24             2215 non-null   float64
 14   pct65up              2215 non-null   float64
 15   persUrban            2215 non-

In [72]:
type(df)

pandas.DataFrame

In [83]:
from tabulate import tabulate

# Reporte de nulos por columna. Con <5% podemos eliminar filas.
# Con >5% habria que evaluar imputacion o eliminar la feature.

def porcentaje_nulos(dataframe):
    '''
        Calcula el porcentaje de nulos de cada feature
        en un dataframe, clasifica aquellas que superan el umbral de
        0.05% y genera dos tablas formateadas, una con un resumen general
        y una lista de alerta.

        Args:
            dataframe (pandas.DataFrame): Dataframe del cual se desea saber
                si existe presencia de nulos.
            
        Returns:
            None: La funcion no retorna ningun valor, imprime el reporte en la salida estandar (consola)
    '''
    nulos = (dataframe.isnull().sum() / len(df) * 100).round(2)

    matriz = []
    revisar = []
    
    for key, value in nulos.items():
        if value > 0.05:
            matriz.append([key, value])
            revisar.append([key, value])
        else:
            matriz.append([key, value])
            

    print(tabulate(matriz, headers=['Feature', 'Porcentaje', 'Revisar']))

    print("\n--Features a revisar por alto porcentaje de nulos--")

    print(f"\n{tabulate(revisar, headers=['Features', 'Porcentaje'])}")

    
        

In [84]:
porcentaje_nulos(df)

Feature                Porcentaje
-------------------  ------------
communityname                0
countyCode                  55.12
communityCode               55.26
fold                         0
State                        0
pop                          0
perHoush                     0
pctBlack                     0
pctWhite                     0
pctAsian                     0
pctHisp                      0
pct12-21                     0
pct12-29                     0
pct16-24                     0
pct65up                      0
persUrban                    0
pctUrban                     0
medIncome                    0
pctWwage                     0
pctWfarm                     0
pctWdiv                      0
pctWsocsec                   0
pctPubAsst                   0
pctRetire                    0
medFamIncome                 0
perCapInc                    0
whitePerCap                  0
blackPerCap                  0
NAperCap                     0
asianPerCap                